## LLM-as-a-Judge Pipeline

**What this notebook does:** Runs one or more LLM judge experiments over a dataset of multi-agent traces and evaluates the predictions against annotations. Reports accuracy, precision, recall, F1, and Cohen's kappa.

**How to use this template:**
1. Copy this folder to a new experiment directory
2. Edit `config.yaml` to define your experiments (one or more)
3. Run this notebook top to bottom

**Inputs:**
- `config.yaml` in the same folder: defines a list of experiments, each with model, backend, dataset path, shots, etc.
- Dataset: set `dataset_path` in config to either `MAD_human_labelled_dataset.json` or `MAD_full_dataset.json`

**Outputs (saved to `saved_results/`):**
- `predictions.csv`  per-trace predictions with token counts and cost for all experiments
- `metrics_per_mode.csv`  per-failure-mode breakdown for all experiments
- `summary.csv`  one row per experiment with overall metrics (accuracy, F1, kappa, 95% CIs)
- `checkpoints/<name>.pkl`  checkpoint saved after every trace (allows resuming)

**Run order:** Top to bottom.

### 1. Setup

Loads `config.yaml`, initialises the `LLMJudge`, and loads the dataset.

In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import json
import os
import sys
from sklearn.metrics import f1_score, precision_score, recall_score, cohen_kappa_score, accuracy_score

# Find repo root regardless of working directory
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "LLM_models_interface")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

from LLM_models_interface.llm_interface import LLMJudge, load_configs, load_dataset

configs = load_configs("config.yaml")

### 2. Run the judge

Iterates over every trace and calls `judge.judge_trace()`. Results are checkpointed to `saved_results/` after each trace, if the run is interrupted, completed results are not lost.

In [2]:
official_modes = ['1.1','1.2','1.3','1.4','1.5',
                  '2.1','2.2','2.3','2.4','2.5','2.6',
                  '3.1','3.2','3.3']

def bootstrap_ci(y_true, y_pred, metric_fn, n=1000, ci=0.95):
        rng = np.random.default_rng(42)
        scores = []
        idx = np.arange(len(y_true))
        yt, yp = np.array(y_true), np.array(y_pred)
        for _ in range(n):
            s = rng.choice(idx, size=len(idx), replace=True)
            scores.append(metric_fn(yt[s], yp[s]))
        lo = np.percentile(scores, (1 - ci) / 2 * 100)
        hi = np.percentile(scores, (1 + ci) / 2 * 100)
        return lo, hi

summary = []
all_metrics_per_mode = []
all_predictions = []

for cfg in configs:
    print(f"\n{'='*60}")
    print(f"Experiment: {cfg.name} — {cfg.model}")
    print(f"{'='*60}")

    judge = LLMJudge(cfg)
    traces = load_dataset(cfg)
    if traces and "round" in traces[0]:
        traces = [t for t in traces if t.get("round") == "Round 3"]

    os.makedirs('saved_results/checkpoints', exist_ok=True)                                                                                                  
    checkpoint_path = f'saved_results/checkpoints/{cfg.name}.pkl'

    results = []
    for record in traces:
        is_full = isinstance(record['trace'], dict)
        trace_id = f"{record['trace']['key']}_{record['trace_id']}" if is_full else record['trace_id']
        trace_text = record['trace']['trajectory'] if is_full else record['trace']

        if len(trace_text) + len(judge.examples) > 1048570:
            trace_text = trace_text[:1048570 - len(judge.examples)]

        try:
            response = judge.judge_trace(trace_id, trace_text)
            results.append(response)
            print(f"  {len(results)}/{len(traces)}")

            with open(checkpoint_path, 'wb') as f:
                pickle.dump(results, f)

            if len(results) % 10 == 0:
                with open(f'saved_results/checkpoints/{cfg.name}_backup_{len(results)}.pkl', 'wb') as f:
                    pickle.dump(results, f)

        except Exception as e:
            print(f"Error on evaluation {len(results)}: {str(e)}")
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(results, f)

    print(f"Progress: {len(results)}/{len(traces)} traces completed")

    rows = [{"trace_id": r.trace_id, "model": r.model_id,
             "name": cfg.name, 
             "tokens_in": r.tokens_in, "tokens_out": r.tokens_out,
             "latency_s": r.latency_s, "cost_usd": r.cost_usd,
             **r.annotations} for r in results]
    all_predictions.extend(rows)

    failure_mode_results = {mode: [r.annotations[mode] for r in results] for mode in official_modes}

    ground_truth = {code: [] for code in official_modes}

    if traces and "mast_annotation" in traces[0]:
        # Full dataset: single annotation per trace
        for record in traces:
            for code in official_modes:
                ground_truth[code].append(record["mast_annotation"].get(code, 0))
    else:
        # Human-labelled dataset: majority vote over 3 annotators
        for record in traces:
            for ann in record["annotations"]:
                code = re.match(r"(\d+\.\d+)", ann["failure mode"]).group(1)
                if code not in official_modes:
                    continue
                votes = [ann["annotator_1"], ann["annotator_2"], ann["annotator_3"]]
                ground_truth[code].append(1 if sum(votes) >= 2 else 0)

    y_true_flat = []
    y_pred_flat = []

    for mode in official_modes:
        y_true_flat.extend(ground_truth[mode])
        y_pred_flat.extend(failure_mode_results[mode])

    print(f"Accuracy: {accuracy_score(y_true_flat, y_pred_flat):.2f}  |  Precision: {precision_score(y_true_flat, y_pred_flat):.2f}  |  Recall: {recall_score(y_true_flat, y_pred_flat):.2f}  |  F1: {f1_score(y_true_flat, y_pred_flat):.2f}  |  Kappa: {cohen_kappa_score(y_true_flat, y_pred_flat):.2f}")

    rows_fm = []                                                                                                                         
    for mode in official_modes:                                                                                                          
        yt = ground_truth[mode]                                                                                                          
        yp = failure_mode_results[mode]                                                                                                  
        rows_fm.append({
            "name": cfg.name,
            "model": cfg.model,                                                                                                                 
            "mode": mode,                                                                                                                
            "precision": precision_score(yt, yp, zero_division=0),
            "recall":    recall_score(yt, yp, zero_division=0),
            "f1":        f1_score(yt, yp, zero_division=0),
            "support":   sum(yt),
        })
    all_metrics_per_mode.extend(rows_fm)

    yt = np.array(y_true_flat)
    yp = np.array(y_pred_flat)

    f1_lo, f1_hi = bootstrap_ci(yt, yp, lambda a, b: f1_score(a, b, zero_division=0))
    kappa_lo, kappa_hi = bootstrap_ci(yt, yp, cohen_kappa_score)

    print(f"F1: {f1_score(yt, yp, zero_division=0):.3f}  95% CI [{f1_lo:.3f}, {f1_hi:.3f}]")
    print(f"Kappa: {cohen_kappa_score(yt, yp):.3f}  95% CI [{kappa_lo:.3f}, {kappa_hi:.3f}]")

    metrics_row = {
        "name": cfg.name, 
        "model": cfg.model,
        "accuracy":  accuracy_score(y_true_flat, y_pred_flat),
        "precision": precision_score(y_true_flat, y_pred_flat),
        "recall":    recall_score(y_true_flat, y_pred_flat),
        "f1":        f1_score(y_true_flat, y_pred_flat, zero_division=0),
        "kappa":     cohen_kappa_score(y_true_flat, y_pred_flat),
        "f1_ci_lo": f1_lo, "f1_ci_hi": f1_hi,
        "kappa_ci_lo": kappa_lo, "kappa_ci_hi": kappa_hi,
        "total_cost_usd":  sum(r.cost_usd for r in results),
        "mean_cost_usd":   sum(r.cost_usd for r in results) / len(results),
        "total_latency_s": sum(r.latency_s for r in results),
        "mean_latency_s":  sum(r.latency_s for r in results) / len(results),
    }
    summary.append(metrics_row)

pd.DataFrame(summary).to_csv("saved_results/summary.csv", index=False)
pd.DataFrame(all_metrics_per_mode).to_csv("saved_results/metrics_per_mode.csv", index=False)
pd.DataFrame(all_predictions).to_csv("saved_results/predictions.csv", index=False)


Experiment: claude-sonnet-4-6_few_shot — claude-sonnet-4-6
No credentials found, authenticating...


c:\Users\bramn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Authenticated with default credentials 
Refreshing credentials...
  1/3
Using cached credentials
  2/3
Using cached credentials
  3/3
Progress: 3/3 traces completed
Accuracy: 0.74  |  Precision: 0.50  |  Recall: 0.18  |  F1: 0.27  |  Kappa: 0.15
F1: 0.267  95% CI [0.000, 0.556]
Kappa: 0.148  95% CI [-0.138, 0.458]

Experiment: gemini-2.5-flash_zero_shot — gemini-2.5-flash
Using cached credentials
  1/3
Using cached credentials
  2/3
Using cached credentials
  3/3
Progress: 3/3 traces completed
Accuracy: 0.71  |  Precision: 0.46  |  Recall: 0.55  |  F1: 0.50  |  Kappa: 0.30
F1: 0.500  95% CI [0.190, 0.710]
Kappa: 0.302  95% CI [-0.046, 0.602]
